### Preload Data and Process

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings("ignore")

In [ ]:
data=pd.read_csv('creditcard.csv')
data.head()

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
duplicates=data.duplicated()
duplicate_index=data[duplicates].index
print("Indicated duplicated rows :", duplicate_index)

In [ ]:
data=data.drop_duplicates()

In [ ]:
data.shape

### Exploratory Data Analysis

In [ ]:
data.describe()

In [ ]:
sns.countplot(x='Class', data=data)
plt.title("Class Distribution (Fraud vs Non-Fraud)")

In [ ]:
fraud =data[data['Class'] == 1]
non_fraud =data[data['Class'] == 0]

fraud['Amount'].hist(alpha=0.5, label='Fraud', bins=50)
non_fraud['Amount'].hist(alpha=0.5, label='Non-Fraud', bins=50)
plt.legend()
plt.title('Amount Distributions: Fraud vs Non-Fraud')

In [ ]:
corr_matrix = data.corr()
print("Correlation Matrix Value :", corr_matrix)

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=False, cmap='RdPu', linewidths=0.5)
plt.title("Correlation Matrix of Features")

### Data Transformation

In [ ]:
features_to_scale=['Amount']
data['Amount'] = np.log(data['Amount'] + 1)

In [ ]:
data.shape

In [ ]:
features = ['V10','V11', 'V12', 'V14', 'V15','V16','V17']
scaler = MinMaxScaler()
data[features] = scaler.fit_transform(data[features])

In [ ]:
data.shape

### Feature Engineering

In [ ]:
X = data.drop('Class', axis=1) 
y = data['Class'] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
# Get feature importance
importance = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)
# Plot feature importance
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
plt.title('Feature Importance from Random Forest')
plt.xlabel('Importance Score')
plt.ylabel('Features')

### Handling Imbalanced Data

In [ ]:
print(data[data['Class'] == 0]['Class'].value_counts())
print(data[data['Class'] == 1]['Class'].value_counts())

In [ ]:
X = data.drop('Class', axis=1)  
y = data['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
smote = SMOTE(sampling_strategy=0.6, random_state=42)

In [ ]:
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
print("Before resampling:", Counter(y_train))
print("After resampling:", Counter(y_resampled))

In [ ]:
data = pd.DataFrame(X_resampled, columns=X.columns) 

#Combine the resampled X and y into a new DataFrame
data['Class'] = y_resampled 

In [ ]:
data.shape #The original shape was (283726,31)

In [ ]:
sns.countplot(x='Class', data=data)
plt.title("Class Distribution (Fraud vs Non-Fraud)")

In [ ]:
print(data.duplicated().sum()) #note that No null values will not appear in the newly generated rows

In [ ]:
print(data[data['Class'] == 0]['Class'].value_counts())
print(data[data['Class'] == 1]['Class'].value_counts())

### Splitting Data

In [ ]:
X = data.drop('Class', axis=1)  
y = data['Class'] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### Model Selection

In [ ]:
classifiers = {
    "LogisticRegression": LogisticRegression(
        penalty='l2',           # L2 regularization to prevent overfitting
        C=2.0,                  # Regularization strength (lower value = stronger regularization)
        solver='liblinear',    
        max_iter=1000           
    ),
    
    "KNearest": KNeighborsClassifier(
         n_neighbors=10,                    
    ),
    
    "XGB Classifier": XGBClassifier(
        max_depth=3,            # Limits the depth of the tree to reduce overfitting
        learning_rate=1,     
        n_estimators=9,      
        subsample=0.8,         
        colsample_bytree=0.8,   
        reg_lambda=1,           # L2 regularization term to prevent overfitting
        reg_alpha=0.2           # L1 regularization term
    ),
    
    "DecisionTreeClassifier": DecisionTreeClassifier(
        max_depth=150,            # Limit depth of the tree to prevent overfitting
        min_samples_split=25,  
        min_samples_leaf=15,     
        max_features='sqrt',   
        ccp_alpha=0.01          
    )
}

### Model Training

In [ ]:
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f"Results for {name}:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, pos_label=1):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred, pos_label=1):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred, pos_label=1):.4f}\n")

for preventing overfitting model : 

In [ ]:
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_train_pred = clf.predict(X_train)
    y_test_pred = clf.predict(X_test)
    
    # Training data performance
    print(f"Training Results for {name}:")
    print(f"Accuracy: {accuracy_score(y_train, y_train_pred):.4f}")
    print(f"Precision: {precision_score(y_train, y_train_pred, pos_label=1):.4f}")
    print(f"Recall: {recall_score(y_train, y_train_pred, pos_label=1):.4f}")
    print(f"F1 Score: {f1_score(y_train, y_train_pred, pos_label=1):.4f}\n")
    
    # Testing data performance
    print(f"Testing Results for {name}:")
    print(f"Accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_test_pred, pos_label=1):.4f}")
    print(f"Recall: {recall_score(y_test, y_test_pred, pos_label=1):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_test_pred, pos_label=1):.4f}\n")

Hyperparameter Tuning, also for preventing overfitting data

In [ ]:
param_grid = {
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['auto', 'sqrt'],
    'ccp_alpha': [0.0, 0.01, 0.1]
}
clf = DecisionTreeClassifier()
grid_search = GridSearchCV(
    clf, 
    param_grid=param_grid, 
    cv=5, 
    scoring='f1', 
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
print(f"Best parameters for DecisionTreeClassifier: {grid_search.best_params_}")
print(f"Best cross-validation score for DecisionTreeClassifier: {grid_search.best_score_:.4f}")

In [ ]:
param_grid = {
   'penalty': ['l1', 'l2'],
        'C': [0.01, 0.1, 1.0, 10.0],
        'solver': ['liblinear'],
        'max_iter': [100, 200]
}
clf = LogisticRegression()
grid_search = GridSearchCV(
    clf, 
    param_grid=param_grid, 
    cv=5, 
    scoring='f1', 
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
print(f"Best parameters for LogisticRegressionClassifier: {grid_search.best_params_}")
print(f"Best cross-validation score for LogisticRegressionClassifier: {grid_search.best_score_:.4f}")

In [ ]:
param_grid = {
   'n_neighbors': [3, 5, 7, 9, 11]
}
clf =KNeighborsClassifier()
grid_search = GridSearchCV(
    clf, 
    param_grid=param_grid, 
    cv=5, 
    scoring='f1', 
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
print(f"Best parameters for KNeighborsClassifier: {grid_search.best_params_}")
print(f"Best cross-validation score for KNeighborsClassifier: {grid_search.best_score_:.4f}")

### Model Evaluation

In [ ]:
# Predictions using the best model
y_pred = classifiers['XGB Classifier'].predict(X_test)

# Evaluate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Model Evaluation Metrics on Test Data:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# ROC-AUC Curve
y_test_proba = classifiers['XGB Classifier'].predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_test_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.4f})")
plt.plot([0,1], [0,1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title("ROC Curve")
plt.legend()
plt.show()